Usei uma fila FIFO, em que o primeiro chamado a entrar é o primeiro a ser atendido. Ela é implementada com uma lista: append() coloca o chamado no final e pop(0) tira do início. Cada chamado é um objeto da classe Chamado (criada com @dataclass), que guarda número, cliente e prioridade juntos, sem precisar escrever o init.

O número do chamado é gerado automaticamente por um contador, o que garante números únicos e em sequência, sem erro de digitação. A prioridade é validada (baixa, normal ou alta) antes de o número ser gerado, então um chamado inválido é recusado sem gastar número. Como o modelo é FIFO, a prioridade é apenas uma informação do chamado e não muda a ordem de atendimento.

In [28]:
"""
Fila de atendimento (FIFO - First In, First Out)

Cada chamado tem: número, cliente e prioridade.
O primeiro chamado a entrar é o primeiro a ser atendido.
O último atendimento pode ser desfeito (o chamado volta para o início da fila).
"""

from dataclasses import dataclass



@dataclass(repr=False)
class Chamado:
    numero: int
    cliente: str
    prioridade: str

    def __repr__(self):
        return f"#{self.numero} {self.cliente} ({self.prioridade})"


class FilaAtendimento:
    def __init__(self):
        self.fila = []
        self.historico = []
        self.proximo_numero = 1

    def abrir_chamado(self, cliente, prioridade="normal"):
        prioridade = prioridade.lower()
        if prioridade not in {"baixa", "normal", "alta"}:
           
            raise ValueError("Prioridade deve ser 'baixa', 'normal' ou 'alta'.")

        chamado = Chamado(self.proximo_numero, cliente, prioridade)
        self.proximo_numero += 1
        self.fila.append(chamado)
        return chamado

    def atender_proximo(self):
        if not self.fila:
            return None

        chamado = self.fila.pop(0)
        self.historico.append(chamado)
        return chamado

    def desfazer_ultimo_atendimento(self):
        if not self.historico:
            return None

        chamado = self.historico.pop()
        self.fila.insert(0, chamado)
        return chamado

    def listar_fila(self):
        return self.fila.copy()


# ---------- Demonstração ----------
central_de_atendimento = FilaAtendimento()

print("Abrindo chamados:")
print("Aberto:", central_de_atendimento.abrir_chamado("Ana", "normal"))
print("Aberto:", central_de_atendimento.abrir_chamado("Bruno", "alta"))
print("Aberto:", central_de_atendimento.abrir_chamado("Carla", "baixa"))
print("Aberto:", central_de_atendimento.abrir_chamado("Daniel", "alta"))

print("\nFila inicial:", central_de_atendimento.listar_fila())

print("\nPróximo atendimento:", central_de_atendimento.atender_proximo())
print("Fila atual:", central_de_atendimento.listar_fila())

print("\nAtendi o chamado errado? Desfazendo...")
print("Desfeito:", central_de_atendimento.desfazer_ultimo_atendimento())
print("Fila atual:", central_de_atendimento.listar_fila())

print("\nAtendendo novamente, agora na ordem correta:")
print(central_de_atendimento.atender_proximo())
print(central_de_atendimento.atender_proximo())
print("Fila atual:", central_de_atendimento.listar_fila())

print("\nHistórico:", central_de_atendimento.historico)

print("\nPrioridade inválida:")
try:
    central_de_atendimento.abrir_chamado("Ana", "urgente")
except ValueError as erro:
    print("Erro:", erro)

print("\nPróximo número livre:", central_de_atendimento.proximo_numero)

Abrindo chamados:
Aberto: #1 Ana (normal)
Aberto: #2 Bruno (alta)
Aberto: #3 Carla (baixa)
Aberto: #4 Daniel (alta)

Fila inicial: [#1 Ana (normal), #2 Bruno (alta), #3 Carla (baixa), #4 Daniel (alta)]

Próximo atendimento: #1 Ana (normal)
Fila atual: [#2 Bruno (alta), #3 Carla (baixa), #4 Daniel (alta)]

Atendi o chamado errado? Desfazendo...
Desfeito: #1 Ana (normal)
Fila atual: [#1 Ana (normal), #2 Bruno (alta), #3 Carla (baixa), #4 Daniel (alta)]

Atendendo novamente, agora na ordem correta:
#1 Ana (normal)
#2 Bruno (alta)
Fila atual: [#3 Carla (baixa), #4 Daniel (alta)]

Histórico: [#1 Ana (normal), #2 Bruno (alta)]

Prioridade inválida:
Erro: Prioridade deve ser 'baixa', 'normal' ou 'alta'.

Próximo número livre: 5
